# MS3 — Step 8: Allele Structural Divergence Clustering

Hierarchical clustering of the 16 HLA alleles by structural signature, using
allele-level mean values of the four features available from Steps 5-7: buried
surface area, contact residue count, polar contact count, and PRODIGY ΔG. This is
the key figure identifying which alleles diverge most structurally despite
presenting similar peptide input -- the manuscript's central "father's question"
figure.

**Honest scope limitation, stated upfront rather than discovered later**: 15 of the
16 alleles in the shortlisted panel have exactly 2 pairs each (HLA-A*02:01 has 5).
Clustering on a mean of 2 observations per allele is thin -- this is an exploratory,
hypothesis-generating figure, not a statistically powered claim about allele-level
structural divergence. This should be stated explicitly in the manuscript alongside
the figure, not left implicit.

**Feature set**: uses the 4 features already verified and exported in Step 7's
scatter data table (buried_surface_area_A2, n_contact_residues, n_polar_contacts,
delta_g_kcal_mol). Step 5's fuller table has additional columns (per-chain contact
breakdown, mean_groove_confidence) that could extend this analysis later if finer
resolution is wanted, but are not required for this clustering.

**Workflow:**
1. Load Step 7's scatter data table (Part A)
2. Aggregate to allele-level mean feature vectors, with per-allele n reported (Part B)
3. Hierarchical clustering (Ward linkage) + dendrogram (Part C)
4. Identify and report the most divergent allele pairs (Part D)

## Part A — Load Step 7's Structure-Affinity Table

In [ ]:
!pip install -q pandas scipy matplotlib 2>/dev/null || pip install -q --break-system-packages pandas scipy matplotlib

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform, pdist

BASE_DIR = Path("/content/ms3_step8_workspace")
TABLES_DIR = BASE_DIR / "tables"
FIGURES_DIR = BASE_DIR / "figures"
for d in (BASE_DIR, TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

REQUIRED_COL = "delta_g_kcal_mol"  # unique to this table, used to verify the right file was uploaded
local_copy = BASE_DIR / "step7_scatter.csv"

while True:
    if not local_copy.exists():
        try:
            from google.colab import files
            print(f"Upload Table_S4_structure_affinity_scatter_data.csv "
                  f"(must contain a '{REQUIRED_COL}' column):")
            uploaded = files.upload()
            for fname, content in uploaded.items():
                if not fname.lower().endswith(".csv"):
                    print(f"  [WRONG FILE TYPE] '{fname}' is not a .csv -- skipping.")
                    continue
                (BASE_DIR / fname).write_bytes(content)
                shutil.copy(BASE_DIR / fname, local_copy)
        except ImportError:
            raise SystemExit("[BLOCKED] Not in Colab and no local copy found.")

    if not local_copy.exists():
        continue

    try:
        scatter_df = pd.read_csv(local_copy)
    except Exception as e:
        print(f"  [PARSE ERROR] {e} -- removing and retrying.")
        local_copy.unlink()
        continue

    if REQUIRED_COL in scatter_df.columns:
        print(f"Verified: loaded correctly ({len(scatter_df)} rows, has '{REQUIRED_COL}')")
        break
    else:
        print(f"  [WRONG FILE] Missing '{REQUIRED_COL}'. Columns found: {list(scatter_df.columns)}")
        local_copy.unlink()

n_alleles = scatter_df["allele"].nunique()
print(f"\nAlleles represented: {n_alleles}")
print(f"Total pairs: {len(scatter_df)}")
scatter_df.head(5)

## Part B — Allele-Level Aggregation

Computes mean feature values per allele, with per-allele n reported explicitly
alongside every row -- so the small-N limitation is visible in the data itself,
not just stated in prose.

In [ ]:
_required = ["scatter_df"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part A first.")

FEATURE_COLS = ["buried_surface_area_A2", "n_contact_residues",
                 "n_polar_contacts", "delta_g_kcal_mol"]

missing_feature_cols = [c for c in FEATURE_COLS if c not in scatter_df.columns]
if missing_feature_cols:
    raise SystemExit(f"[BLOCKED] scatter_df is missing required feature column(s): "
                     f"{missing_feature_cols}")

allele_agg = scatter_df.groupby("allele").agg(
    n_pairs=("pair_id", "count"),
    locus=("locus", "first"),
    is_class_ii=("is_class_ii", "first"),
    **{f"mean_{c}": (c, "mean") for c in FEATURE_COLS}
).reset_index()

print(f"Aggregated to {len(allele_agg)} alleles")
print(f"\nPer-allele sample size (n_pairs):")
print(allele_agg[["allele", "n_pairs"]].to_string(index=False))

low_n_alleles = allele_agg[allele_agg["n_pairs"] < 3]["allele"].tolist()
if low_n_alleles:
    print(f"\n[NOTE] {len(low_n_alleles)} allele(s) have fewer than 3 pairs "
          f"({low_n_alleles}) -- their position in the dendrogram below should be "
          f"read as exploratory, not as a statistically robust structural signature.")

allele_agg

## Part C — Hierarchical Clustering and Dendrogram

Ward linkage on standardized (z-scored) allele-level mean features, so BSA (scale
~1700-2400) doesn't dominate the distance metric over contact counts (scale ~25-45)
purely due to differing units.

In [ ]:
_required = ["allele_agg", "FEATURE_COLS"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part B first.")

feature_matrix = allele_agg[[f"mean_{c}" for c in FEATURE_COLS]].values
allele_labels = allele_agg["allele"].tolist()

# Standardize each feature (z-score) before clustering, so features with
# larger raw scales (BSA in the thousands) don't dominate the Euclidean
# distance used by Ward linkage purely due to units, not genuine divergence
feature_mean = feature_matrix.mean(axis=0)
feature_std = feature_matrix.std(axis=0)
feature_std[feature_std == 0] = 1  # avoid divide-by-zero for any constant column
standardized = (feature_matrix - feature_mean) / feature_std

Z = linkage(standardized, method="ward")

fig, ax = plt.subplots(figsize=(10, 6))
dendro = dendrogram(
    Z, labels=allele_labels, ax=ax,
    leaf_rotation=90, color_threshold=0.7 * max(Z[:, 2]),
)
ax.set_title("Allele Structural Divergence\n(Ward linkage, standardized BSA + "
             "contacts + polar contacts + PRODIGY dG)")
ax.set_ylabel("Distance")
plt.tight_layout()

fig_path = FIGURES_DIR / "Figure_allele_structural_dendrogram.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"\nSaved: {fig_path}")
print("\n[REMINDER] This dendrogram is built from allele-level means with n=2 for "
      "15/16 alleles -- present it as an exploratory structural map, and report the "
      "per-allele n from Part B alongside it, not as a definitively powered result.")

## Part D — Most Divergent Allele Pairs

In [ ]:
_required = ["standardized", "allele_labels"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part C first.")

# Pairwise Euclidean distance in standardized feature space -- identifies
# which specific allele pairs are most structurally divergent, the direct
# answer to "which alleles differ most despite similar peptide input"
dist_matrix = squareform(pdist(standardized, metric="euclidean"))
dist_df = pd.DataFrame(dist_matrix, index=allele_labels, columns=allele_labels)

# Extract the upper triangle as a flat, sorted list of pairs
pairs = []
for i in range(len(allele_labels)):
    for j in range(i + 1, len(allele_labels)):
        pairs.append({
            "allele_1": allele_labels[i],
            "allele_2": allele_labels[j],
            "structural_distance": round(dist_matrix[i, j], 3),
        })
pairs_df = pd.DataFrame(pairs).sort_values("structural_distance", ascending=False)

print("Top 10 most structurally divergent allele pairs:")
print(pairs_df.head(10).to_string(index=False))

print("\nTop 10 most structurally similar allele pairs:")
print(pairs_df.tail(10).sort_values("structural_distance").to_string(index=False))

# Export everything
allele_agg_path = TABLES_DIR / "Table_S5_allele_structural_aggregation.csv"
pairs_path = TABLES_DIR / "Table_S5_allele_pairwise_distances.csv"
allele_agg.to_csv(allele_agg_path, index=False)
pairs_df.to_csv(pairs_path, index=False)

print(f"\nSaved: {allele_agg_path}")
print(f"Saved: {pairs_path}")

try:
    from google.colab import files
    files.download(str(allele_agg_path))
    files.download(str(pairs_path))
    files.download(str(FIGURES_DIR / "Figure_allele_structural_dendrogram.png"))
except ImportError:
    print("Files saved locally.")

print("\nStep 8 complete. Feed the dendrogram and divergent-pair table into the "
      "manuscript's results/discussion; carry forward the small-N caveat explicitly.")

## Part E — Within-Class Clustering Check (chain-count artifact test)

**Why this cell exists**: class II complexes have two HLA chains contacting the peptide (DRA+DRB1); class I complexes have one (the heavy chain only). A larger structural interface for class II is therefore expected from chain count alone, independent of any genuine allele-level signal. Table_S5's allele-level means show zero overlap between class I and class II on every feature, and the four largest pairwise distances in the dataset are all class-I-vs-class-II pairs -- consistent with either a real biological signal or a trivial chain-count artifact, and this cell is what distinguishes between them.

This re-runs Ward clustering **separately within each class**, after re-standardizing features within that class only (so one class's larger raw scale cannot leak into the other class's z-scores). If class I alleles (or class II alleles) show no meaningful substructure among themselves, the I/II bifurcation in the combined dendrogram should be presented as consistent with the chain-count difference, not as evidence of allelic structural divergence beyond that.

In [ ]:
_required = ["allele_agg", "FEATURE_COLS"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part B first.")

def cluster_within_class(agg_df, class_mask, label):
    sub = agg_df[class_mask].copy()
    n = len(sub)
    if n < 3:
        print(f"{label}: n={n} alleles, too few to cluster meaningfully (need >=3).")
        return None

    feat_matrix = sub[[f"mean_{c}" for c in FEATURE_COLS]].values
    feat_mean = feat_matrix.mean(axis=0)
    feat_std = feat_matrix.std(axis=0)
    feat_std[feat_std == 0] = 1
    standardized_sub = (feat_matrix - feat_mean) / feat_std

    Z_sub = linkage(standardized_sub, method="ward")
    labels_sub = sub["allele"].tolist()

    fig, ax = plt.subplots(figsize=(8, 4))
    dendrogram(Z_sub, labels=labels_sub, ax=ax, leaf_rotation=90)
    ax.set_title(f"Within-{label} Allele Clustering\n(Ward linkage, re-standardized within class)")
    ax.set_ylabel("Distance")
    plt.tight_layout()
    fig_path = FIGURES_DIR / f"Figure_within_{label.lower().replace(' ', '_')}_clustering.png"
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()

    dist_sub = squareform(pdist(standardized_sub, metric="euclidean"))
    max_dist = dist_sub[np.triu_indices(n, k=1)].max() if n > 1 else None
    min_dist = dist_sub[np.triu_indices(n, k=1)].min() if n > 1 else None

    print(f"{label}: n={n} alleles, {len(FEATURE_COLS)} features")
    print(f"  Within-class pairwise distance range: {min_dist:.3f} - {max_dist:.3f}")
    print(f"  Saved: {fig_path}")
    return {"label": label, "n": n, "min_dist": min_dist, "max_dist": max_dist, "fig_path": str(fig_path)}

class_i_mask = allele_agg["is_class_ii"] == False
class_ii_mask = allele_agg["is_class_ii"] == True

results_within = []
r1 = cluster_within_class(allele_agg, class_i_mask, "Class I")
if r1:
    results_within.append(r1)
r2 = cluster_within_class(allele_agg, class_ii_mask, "Class II")
if r2:
    results_within.append(r2)

if results_within:
    combined_max = max(r["max_dist"] for r in results_within)
    cross_class_max = pairs_df["structural_distance"].max() if "pairs_df" in dir() else None
    print(f"\nLargest within-class pairwise distance (either class): {combined_max:.3f}")
    if cross_class_max is not None:
        print(f"Largest cross-class pairwise distance (from Part D): {cross_class_max:.3f}")
        ratio = cross_class_max / combined_max if combined_max > 0 else float("inf")
        print(f"Cross-class / within-class max-distance ratio: {ratio:.2f}x")
        print("\n[INTERPRETATION] If this ratio is large, the combined dendrogram's I/II "
              "split is dominated by the class-level (chain-count) difference, and any "
              "within-class structure here -- not the I/II split itself -- is the part "
              "that can be read as genuine allelic divergence. State this explicitly "
              "alongside the combined dendrogram in the manuscript.")

within_class_summary_path = TABLES_DIR / "Table_S5b_within_class_clustering_summary.csv"
pd.DataFrame(results_within).to_csv(within_class_summary_path, index=False)
print(f"\nSaved: {within_class_summary_path}")
